# A pure-Python UCBBuilder (issue #72)

This notebook shows how to reimplement the helper from [synth-learn/UCBBuilder.py](https://github.com/mrudu/synth-learn/blob/master/LTLsynthesis/UCBBuilder.py) **without shelling out to the acacia binary** — everything runs in-process through the `acacia_boomslang` bindings.

| synth-learn reference                       | `acacia_boomslang` equivalent                                          |
|---------------------------------------------|---------------------------------------------------------------------|
| `subprocess.run(acacia-bonsai ...)`         | `create_twa` + `solve_acacia_safety_game`                           |
| parsing the `ANTICHAIN` lines of stdout     | iterate `WinningRegion` / call `.contains(vec)`                     |
| parsing the `AUTOMATA` section              | `get_aut_hoa` + `spot.automaton`                                    |
| `self.num_states` / `ucb.state_is_accepting`| `num_states(game)` / `state_is_accepting(game, s)`                  |
| `get_transition_state(v, edge_label)`       | `successor(game, v, true_aps, false_aps, k)`                        |
| `is_safe(v)`                                | `WinningRegion.contains(v)`                                         |

In [ ]:
import itertools
from dataclasses import dataclass
from typing import Iterable, List, Optional

import acacia_boomslang as ab

## The `UCB` class

A dataclass that bundles the game, the winning region (playing the role of the reference’s `antichain_heads`) and a handful of helper methods with the same names as the ones in synth-learn. `build()` mirrors `build_UCB` by retrying with larger $k$ until the spec becomes realisable (or we run out of budget).

In [ ]:
@dataclass
class UCB:
    k: int
    psi: str
    inputs: List[str]
    outputs: List[str]
    game: "ab.Game"
    winreg: "ab.WinningRegion"
    num_states: int

    @classmethod
    def build(cls, psi, inputs, outputs, k=2, limit=10) -> Optional["UCB"]:
        while k <= limit:
            game = ab.create_twa(psi, inputs, outputs)
            ab.preprocess_aut_standard(game, k_max=k)
            ab.set_bool_thresh_no_bool_states(game, k_max=k)
            result = ab.solve_acacia_safety_game(
                game, k_max=k, k_min=min(2, k), k_inc=1)
            if result.is_real():
                return cls(
                    k=k, psi=psi, inputs=inputs, outputs=outputs,
                    game=game, winreg=result.get_winning_region(),
                    num_states=ab.num_states(game))
            k += 1
        return None

    def initial_state_vector(self) -> List[int]:
        v = ab.get_initial_state(self.game)
        return [v[i] for i in range(len(v))]

    def get_transition_state(self, state_vector, true_aps, false_aps):
        v = ab.make_vector(self.game, ab.IntVector(state_vector))
        s = ab.successor(self.game, v,
                         ab.StringVector(list(true_aps)),
                         ab.StringVector(list(false_aps)),
                         self.k)
        return [s[i] for i in range(len(s))]

    def is_safe(self, state_vector) -> bool:
        v = ab.make_vector(self.game, ab.IntVector(state_vector))
        return self.winreg.contains(v)

    def iter_io_cubes(self):
        all_aps = self.inputs + self.outputs
        for bits in itertools.product([False, True], repeat=len(all_aps)):
            t = [a for a, b in zip(all_aps, bits) if b]
            f = [a for a, b in zip(all_aps, bits) if not b]
            yield t, f

## Build a UCB for the GR(1) spec

In [ ]:
psi = "!((G (F (req))) -> (G (F (grant))))"
ucb = UCB.build(psi, ["req"], ["grant"], k=2)
assert ucb is not None
print(f"built UCB at k={ucb.k}: {ucb.num_states} states, {len(ucb.winreg)} antichain heads")

## Forward-simulate one step for every IO cube

This is the typical usage pattern from synth-learn: at a given state vector, enumerate every IO assignment and compute the resulting state vector, keeping the ones still inside the winning region.

In [ ]:
def format_cube(t, f):
    return " & ".join([*t, *(f"!{a}" for a in f)]) or "tt"

v0 = ucb.initial_state_vector()
print(f"initial state vector: {v0}  is_safe={ucb.is_safe(v0)}")
print()
print("IO cube         -> next state          safe?")
print("-" * 50)
for t, f in ucb.iter_io_cubes():
    nxt = ucb.get_transition_state(v0, t, f)
    print(f"{format_cube(t, f):>14}  -> {str(nxt):>18}   {ucb.is_safe(nxt)}")

## Sanity check against the underlying TWA

The `successor` helper implements the exact same forward update as the solver’s standard actioner. We can double-check by running an independent simulation through the `spot` + `buddy` bindings and making sure the two match.

In [ ]:
import spot
import buddy

aut = spot.automaton(ab.get_aut_hoa(ucb.game))
ap_vars = {p: buddy.bdd_ithvar(aut.register_ap(p)) for p in ucb.inputs + ucb.outputs}

def ref_successor(v, t, f, k_cap):
    cube = buddy.bddtrue
    for a in t:
        cube &= ap_vars[a]
    for a in f:
        cube &= buddy.bdd_not(ap_vars[a])
    out = [-1] * ucb.num_states
    for src in range(ucb.num_states):
        if v[src] == -1:
            continue
        for e in aut.out(src):
            if (e.cond & cube) == buddy.bddfalse:
                continue
            acc = 1 if aut.state_is_accepting(e.dst) else 0
            cand = min(k_cap, v[src] + acc)
            if cand > out[e.dst]:
                out[e.dst] = cand
    return out

v = [0] * ucb.num_states
for t, f in ucb.iter_io_cubes():
    mine = ucb.get_transition_state(v, t, f)
    ref  = ref_successor(v, t, f, ucb.k)
    assert mine == ref, (t, f, mine, ref)
print("every IO cube agrees with the spot/buddy reference simulation")